- select a robust universe of ortagonal return sourcet e.g. (Bonds, Commodity ETFs, by sector ETFs, etc...
- filter by orthagonality/correlation
- from remaining assets construct a portfolio using Entropy pooling
- test on historical returns(get ES data from the most recent market crash)



In [199]:
pip install ecos

In [200]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
import cvxpy as cp

In [201]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue
            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()
            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      tickers:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = yf.download(tickers, start, end, interval)["Close"]
      bench_data = yf.download([benchmark], start, end, interval)["Close"]

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data= pd.read_parquet(benchmark_path)

    returns_raw = df.pct_change().dropna()
    benchmark = bench_data.pct_change().dropna()
    self.universe = returns_raw.columns

    return returns_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [73]:
class Optimizer:
  def __init__(
      self,
      optim_params,
      debug:bool=False,
      **kwargs
  ):
    self.debug = debug
    self.es_prct = optim_params.es_prct
    self.turnover_penalty = optim_params.turnover_penalty
    self.risk_pen = optim_params.risk_penalty
    self.tail_penalty = optim_params.tail_penalty

  def optimize_lambda(self, p, A, b, k_eq, k_ineq):
    constraints = []
    K = k_eq + k_ineq
    lambda_var = cp.Variable(K)

    if k_ineq > 0:
        constraints.append(lambda_var[k_eq:] >= 0)

    exp_terms = -lambda_var @ A + np.log(p)
    obj_fn = cp.log_sum_exp(exp_terms) + lambda_var @ b

    prob = cp.Problem(cp.Minimize(obj_fn), constraints)
    prob.solve(solver=cp.ECOS)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimization failed. Status: {prob.status}")

    return lambda_var.value

  def optimize_w(self, S, N, returns, mu, cov, w_prev=None):
    R = returns.values
    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    es = zeta + (1 / ((1-self.es_prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0
    ]

    ex_r = mu @ w

    if turnover_penalty is not None and w_prev is not None:
      turnover_penalty = self.turnover_penalty * cp.sum((w - w_prev)**2)

    else:
      turnover_penalty = 0

    risk_term = cp.quad_form(w, cov)
    obj_fn = cp.Maximize(ex_r - risk_term - es - turnover_penalty)

    prob = cp.Problem(obj_fn, constraints)
    prob.solve(solver=cp.CLARABEL)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"GMV optimization failed: {prob.status}")

    return w.value







In [190]:
class Portfolio(Optimizer, DataStore):
  def __init__(self, optim_params, debug:bool=False, **kwargs):
    self.debug = debug
    super().__init__(debug=debug, optim_params=optim_params, **kwargs)

  def _add(self, A, b, vtype):
    if vtype == "ineq":
      self.A_ineq.append(A)
      self.b_ineq.append(b)
    elif vtype == "eq":
      self.A_eq.append(A)
      self.b_eq.append(b)

  def get_views(self, views, returns, p):
    self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = [], [], [], []

    R = returns.values
    for view in views:
      if view["view"] == "mean":
        idx = returns.columns.get_loc(view["ticker"])

        self._add(R[:, idx], view["value"], view["type"])

      elif view["view"] == "volatility":
        idx = returns.columns.get_loc(view["ticker"])
        R_ = R[:, idx]
        mu = np.sum(p*R_)
        var = (R_ - mu)**2

        self._add(var, view["value"]**2, view["type"])

      elif view["view"] == "relative":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        A_rel = R[:, idx1] - R[:, idx2]

        self._add(A_rel, view["value"], view["type"])

      elif view["view"] == "correlation":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        mu_1 = np.sum(p*R[:, idx1])
        mu_2 = np.sum(p*R[:, idx2])

        vol_1 = np.sqrt(np.sum(p*(R[:, idx1] - mu_1)**2))
        vol_2 = np.sqrt(np.sum(p*(R[:, idx2] - mu_2)**2))
        rho = view["value"]
        cov_rarget = rho*vol_1*vol_2

        A_corr = (R[:, idx1] - mu_1)*(R[:, idx2] - mu_2)
        self._add(A_corr, cov_rarget, view["type"])



  def solve_entropy_pooling(
      self,
      R,
      views=None,
      p=None
  ):
    S, N = R.shape
    if p is None:
      p = np.ones(S) / S
    if views is not None:
        self.get_views(views, R, p)

    else:
      self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = None, None, None, None

    A_list, b_list, constraints = [], [], []
    k_eq = 0

    if self.A_eq is not None and self.b_eq is not None:
      A_eq_clean = np.atleast_2d(self.A_eq)
      b_eq_clean = np.atleast_1d(self.b_eq)
      A_list.append(A_eq_clean)
      b_list.append(b_eq_clean)
      k_eq = A_eq_clean.shape[0]

    k_ineq = 0
    if self.A_ineq is not None and self.b_ineq is not None:
      A_ineq_clean = np.atleast_2d(self.A_ineq)
      b_ineq_clean = np.atleast_1d(self.b_ineq)
      A_list.append(A_ineq_clean)
      b_list.append(b_ineq_clean)
      k_ineq = A_ineq_clean.shape[0]

    A = np.vstack(A_list)
    b = np.concatenate(b_list)

    opt_lambda = self.optimize_lambda(p, A, b, k_eq, k_ineq)

    q = p * np.exp(-opt_lambda @ A)
    q /= np.sum(q)
    mu_post = q @ R
    R_dev = R - mu_post
    cov_post = (R_dev.T * q) @ R_dev

    return mu_post, cov_post

  def optimize_portfolio(self, returns, views, p=None):
    S, N = returns.shape

    p = p if p is not None else np.ones(S)/S
    mu_post, cov_post = self.get_views(views, returns, p)
    w_opt = self.optimize_w(S, N, returns, mu_post, cov_post)

    return w_opt






In [191]:
from dataclasses import dataclass, asdict
@dataclass
class OptimParams:
  es_prct: float
  turnover_penalty: float
  risk_penalty: float
  tail_penalty: float


In [192]:
views = [
    # -------------------------------------------------------------------------
    # 1. MEAN VIEWS (Absolute Expected Returns)
    # -------------------------------------------------------------------------
    {
        "view": "mean",
        "ticker": "QLD",
        "value": 0.08,
        "type": "eq"
    },
    {
        "view": "mean",
        "ticker": "KMLM",
        "value": 0.12,
        "type": "ineq"
    },

    # -------------------------------------------------------------------------
    # 2. VOLATILITY VIEWS (Absolute Standard Deviation)
    # -------------------------------------------------------------------------
    {
        "view": "volatility",
        "ticker": "QLD",
        "value": 0.35,
        "type": "eq"
    },
    {
        "view": "volatility",
        "ticker": "BNDW",
        "value": 0.08,
        "type": "ineq"
    },

    # -------------------------------------------------------------------------
    # 3. RELATIVE VIEWS (Spread Outperformance)
    # -------------------------------------------------------------------------
    {
        "view": "relative",
        "ticker1": "KMLM",
        "ticker2": "BNDW",
        "value": 0.04,
        "type": "ineq"
    },

    # -------------------------------------------------------------------------
    # 4. CORRELATION VIEWS (Co-movements)
    # -------------------------------------------------------------------------
    {
        "view": "correlation",
        "ticker1": "QLD",
        "ticker2": "BNDW",
        "value": 0.25,
        "type": "eq"
    },
    {
        "view": "correlation",
        "ticker1": "QLD",
        "ticker2": "KMLM",
        "value": -0.40,
        "type": "eq"
    }
]

In [193]:
tickers = [
  "QLD",
  "BNDW",
  "KMLM"
]

optim_params = OptimParams(
    es_prct=0.95,
    turnover_penalty=1,
    risk_penalty=2,
    tail_penalty=1.5
)

In [194]:
ptf = Portfolio(optim_params, debug=False)

In [195]:
returns, benchmark = ptf.get_data(
    tickers=tickers,
    start="2020-01-01",
    end="2026-01-01",
    interval="1d"
)

In [ ]:
ptf.solve_entropy_pooling(R=returns, views=views)